In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild from the raw CSV through cleaning, the split, and every embedding, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_cleaned, df_engineered)
    del df_raw, df_cleaned, df_engineered


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import fit_phrases

if REBUILD:
    embedding_model = EmbeddingModel(df_train, df_valid, df_test, fit_phrases(df_train))
    for name, train_function in EmbeddingModel.train_registry.items():
        print(f"Embedding Model `{name}` Training")
        train_function(embedding_model)
    for name, predict_function in EmbeddingModel.predict_registry.items():
        print(f"Embedding Model `{name}` Running")
        with data_oi.scope():
            train_embedding, valid_embedding, test_embedding = predict_function(embedding_model)
            data_oi.save_embedding_data(
                name,
                train_embedding,
                valid_embedding,
                test_embedding,
            )
            data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
            del train_embedding, valid_embedding, test_embedding
    for extra in (embedding_model._train_expanded, embedding_model._train_expanded_phrase):
        if extra is not None:
            data_oi.unpersist(extra)
    data_oi.unpersist(df_train, df_valid, df_test)
    del embedding_model, df_train, df_valid, df_test


In [ ]:
from src.analytics import AnalyticsModel
from src.analytics.mlp import DEFAULT_PARA, MLPTrainer

from copy import deepcopy

import numpy as np
from sklearn.metrics import f1_score

# Use the public validation split here; do not run KFold or an optimizer search.
para = {**DEFAULT_PARA, "epochs": 100, "seed": SEED}
PATIENCE = 7
MIN_DELTA = 0.0001
# Each variant specifies (hidden layers, use training weights).
VARIANTS = {
    "a": ((64,), False),
    "b": ((64,), True),
    "c": ((64, 32), False),
    "d": ((64, 32), True),
}


def fit_with_validation(train, valid_rows, labels, hidden_layers, weighted):
    trainer = MLPTrainer(train, hidden_layers, weighted, **para)
    history = []
    best_score = last_improvement = -float("inf")
    stale_epochs = 0
    for epoch in range(1, trainer.epochs + 1):
        loss = trainer.train_epoch()
        predictions = trainer.predict_rows(valid_rows)
        score = float(f1_score(labels, predictions, average="weighted", zero_division=0))
        history.append({"epoch": epoch, "train_loss": loss, "valid_weighted_f1": score})
        # Save every new best score, even if it does not reset patience.
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = deepcopy(trainer.network.state_dict())
        if score > last_improvement + MIN_DELTA:
            last_improvement, stale_epochs = score, 0
        else:
            stale_epochs += 1
        print(f"Epoch {epoch}: loss {loss:.5f}, validation F1 {score:.5f}")
        if stale_epochs >= PATIENCE:
            break
    trainer.network.load_state_dict(best_state)
    return trainer, history, best_epoch


In [ ]:
import json
from datetime import datetime

import pandas as pd
from src.analytics.mlp import FEATURES
from src.paths import TEMP_DIR

run_dir = TEMP_DIR / "mlp" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True)
settings = {"purpose": "validation_selection", "embeddings": list(EmbeddingModel.predict_registry),
            "split_seed": SEED, "para": para, "variants": VARIANTS,
            "patience": PATIENCE, "min_delta": MIN_DELTA}
(run_dir / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")

train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}
results, epoch_history = [], []

for embedding_name in EmbeddingModel.predict_registry:
    train_embedding, valid_embedding, test_embedding = data_oi.load_embedding_data(embedding_name)
    analytics_model = AnalyticsModel(
        embedding_name,
        train_embedding,
        valid_embedding,
        test_embedding,
    )
    train_df, valid_df, test_df = analytics_model._feature_transform(FEATURES)
    # Cache only assembled inputs, not a second copy of all embedding columns.
    analytics_model.release_features()
    data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
    train_df = train_df.select("uniqueID", "rating", "rating_p1", "weight", "features").cache()
    valid_df = valid_df.select("uniqueID", "rating", "rating_p1", "features").cache()
    test_df = test_df.select("uniqueID", "rating", "features").cache()
    valid_rows = list(valid_df.select("rating_p1", "features").toLocalIterator())
    labels = np.array([row.rating_p1 for row in valid_rows], dtype=np.int64)
    for variant, (hidden_layers, weighted) in VARIANTS.items():
        analytics_name = f"mlp_{variant}"
        print(f"Embedding Model `{embedding_name}` Analytics Model `{analytics_name}` Running")
        # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
        model_name = f"{analytics_name}__{embedding_name}"
        # Early stopping uses validation here; the registered mlp() fits on train only.
        trainer, history, best_epoch = fit_with_validation(train_df, valid_rows, labels, hidden_layers, weighted)
        train_pred = trainer.predict(train_df)
        valid_pred = trainer.predict(valid_df)
        test_pred = trainer.predict(test_df)
        train_pred, valid_pred, test_pred = train_pred.cache(), valid_pred.cache(), test_pred.cache()
        train_pred.count()
        valid_pred.count()
        test_pred.count()
        train_pred_dict[model_name] = train_pred
        valid_pred_dict[model_name] = valid_pred
        test_pred_dict[model_name] = test_pred
        del trainer

        tags = {"Embedding_Model": embedding_name, "Analytic_Model": analytics_name}
        epoch_history.extend({**tags, **row} for row in history)
        best_f1 = history[best_epoch - 1]["valid_weighted_f1"]
        results.append({**tags, "best_epoch": best_epoch, "epochs_run": len(history),
                        "best_valid_weighted_f1": best_f1})
        pd.DataFrame(epoch_history).to_csv(run_dir / "history.csv", index=False)
        pd.DataFrame(results).to_csv(run_dir / "summary.csv", index=False)
        print(f"{model_name}: epoch {best_epoch}/{len(history)}, validation F1 {best_f1:.5f}")
    data_oi.unpersist(train_df, valid_df, test_df)
    del train_embedding, valid_embedding, test_embedding, analytics_model, train_df, valid_df, test_df, valid_rows

summary = pd.DataFrame(results)
display(summary)


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)

scorer_train.score.toPandas().to_csv(run_dir / "train_metrics.csv", index=False)
scorer_valid.score.toPandas().to_csv(run_dir / "valid_metrics.csv", index=False)

scorer_test.score.toPandas().to_csv(run_dir / "test_metrics.csv", index=False)

for predictions in (train_pred_dict, valid_pred_dict, test_pred_dict):
    data_oi.unpersist(*predictions.values())

# One shared architecture; ties prefer A, B, C, then D.
comparison = summary.groupby("Analytic_Model", as_index=False).agg(
    mean_valid_weighted_f1=("best_valid_weighted_f1", "mean"),
    median_best_epoch=("best_epoch", "median"),
).sort_values(["mean_valid_weighted_f1", "Analytic_Model"], ascending=[False, True])
display(comparison)
comparison.to_csv(run_dir / "comparison.csv", index=False)

winner = comparison.iloc[0]
variant = winner["Analytic_Model"].removeprefix("mlp_")
hidden_layers, weighted = VARIANTS[variant]
# Median is a fixed-budget recommendation, not an evaluated common-epoch score.
proposed_defaults = {
    "HIDDEN_LAYERS": hidden_layers,
    "USE_WEIGHTS": weighted,
    "DEFAULT_PARA": {**para, "epochs": int(winner["median_best_epoch"])},
}
(run_dir / "proposed_defaults.json").write_text(
    json.dumps(proposed_defaults, indent=2), encoding="utf-8"
)
print(proposed_defaults)
# Copy these values into mlp.py after review, then run the public mlp entry for final scores.
# Test metrics are for comparison only; selection uses validation and leaves defaults unchanged.
